# 02 Baselines and modelling
Every number here is read from the experiment tracker, so each result traces to a logged run.

Make the project folder importable.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

Import libraries and project modules.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config, data, evaluate

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import joblib
import mlflow
from mlflow.tracking import MlflowClient
from sklearn import set_config

set_config(display="text")

Connect to the tracker and select the latest training session.

In [ ]:
mlflow.set_tracking_uri(config.TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[config.EXPERIMENT_NAME], order_by=["attributes.start_time ASC"])
if runs.empty:
    raise RuntimeError("No tracked runs found. Run python src/train.py first.")
session = runs["tags.session_id"].dropna().max()
runs = runs[runs["tags.session_id"] == session].copy()
runs["run"] = runs["tags.mlflow.runName"]
print("session:", session, "| runs:", len(runs))
print("data md5:", runs["tags.data_md5"].iloc[0])
print("git commit:", runs["tags.git_commit"].iloc[0])

Build the baseline-versus-model comparison table from the tracked runs.

In [ ]:
cv = runs[runs["run"] != "final-model"].copy()
columns = {
    "run": "run",
    "metrics.cv_pr_auc_mean": "pr_auc_mean",
    "metrics.cv_pr_auc_std": "pr_auc_std",
    "metrics.cv_precision_at_flag_mean": "precision_mean",
    "metrics.cv_precision_at_flag_std": "precision_std",
    "metrics.cv_lift_at_flag_mean": "lift_mean",
    "metrics.cv_roc_auc_mean": "roc_auc_mean",
}
comparison = cv[list(columns)].rename(columns=columns).set_index("run").round(4)
display(comparison)

Plot cross-validated PR-AUC and precision at the flag rate with fold spread.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison["pr_auc_mean"].plot.barh(xerr=comparison["pr_auc_std"], ax=axes[0], title="CV PR-AUC (mean, std)")
comparison["precision_mean"].plot.barh(xerr=comparison["precision_std"], ax=axes[1], title="CV precision at flag rate (mean, std)")
fig.tight_layout()
plt.show()

Plot per-fold PR-AUC to see the spread across forward-chaining folds.

In [ ]:
client = MlflowClient()
fig, ax = plt.subplots(figsize=(8, 4))
for _, row in cv.iterrows():
    history = sorted(client.get_metric_history(row["run_id"], "cv_fold_pr_auc"), key=lambda m: m.step)
    ax.plot([m.step + 1 for m in history], [m.value for m in history], marker="o", label=row["run"])
ax.set_xlabel("fold")
ax.set_ylabel("PR-AUC")
ax.legend()
plt.show()

Show the tuned hyperparameters of each tuned model.

In [ ]:
skip = {"seed", "split_strategy", "cv_strategy", "flag_rate", "decision_point", "train_rows", "feature_set", "model"}
for _, row in cv[cv["run"].str.endswith("-tuned")].iterrows():
    params = client.get_run(row["run_id"]).data.params
    print(row["run"], {k: v for k, v in params.items() if k not in skip})

Show which run became the final model and confirm the registry points at it.

In [ ]:
final = runs[runs["run"] == "final-model"].iloc[0]
final_params = client.get_run(final["run_id"]).data.params
print("final run id:", final["run_id"])
print("selected run:", final_params["selected_run"])
print("selection rule:", final_params["selection_rule"])
print("threshold:", final_params["threshold"])
version = client.get_model_version_by_alias(config.MODEL_NAME, "final")
print("registry version:", version.version, "| registry run id matches:", version.run_id == final["run_id"])

Show the structure of the saved pipeline.

In [ ]:
pipeline = joblib.load(config.MODEL_PATH)
print(pipeline)

Reload the saved model and confirm it reproduces the predictions already logged by training.

In [ ]:
table, _ = data.build_model_table()
logged = pd.read_csv(config.REPORTS_DIR / "test_predictions.csv")
test = table[table["split"] == "test"].merge(logged[["appointment_id", "probability"]], on="appointment_id")
fresh = pipeline.predict_proba(test[config.RAW_INPUT_COLUMNS])[:, 1]
print("rows compared:", len(test))
print("max absolute difference:", float(np.abs(fresh - test["probability"].to_numpy()).max()))
print("identical:", bool(np.allclose(fresh, test["probability"].to_numpy(), atol=1e-9)))

## Findings
Write 4 to 6 sentences here after running: whether the models beat the rule and the simple baseline, the fold spread, and why the selected family was chosen.